Total training compute(in GFLOPs) for models found on the pareto frontier(Scaling laws for robust comparison).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

C = np.array([7.05e+6, 1.69e+7, 3.5e+7, 7.05e+7, 1.24e+8, 1.69e+8])  # considering first 7 models
Cmin, Cmax = C.min(), C.max()

target_logs = np.linspace(np.log10(Cmin), np.log10(Cmax), 5)

plt.figure()
plt.xscale("log")
plt.scatter(np.log10(C), np.zeros_like(C), label="Actual models")
plt.scatter(target_logs, np.zeros_like(target_logs), marker="x", label="Target log points")

plt.yticks([])
plt.xlabel("log10(GFLOPs)")
plt.legend()
plt.title("Actual compute vs. 5 target log-spaced baseline points")

plt.show()

C = np.array([7.05e+6, 1.5e+7, 1.69e+7, 3.5e+7, 7.05e+7, 1.24e+8, 1.69e+8, 3.52e+8,
              6.23e+8])  # considering first 9 models, assuming we have 64m samples
Cmin, Cmax = C.min(), C.max()

target_logs = np.linspace(np.log10(Cmin), np.log10(Cmax), 5)
plt.figure()
plt.xscale("log")
plt.scatter(np.log10(C), np.zeros_like(C), label="Actual models")
plt.scatter(target_logs, np.zeros_like(target_logs), marker="x", label="Target log points")

plt.yticks([])
plt.xlabel("log10(GFLOPs)")
plt.legend()
plt.title("Actual compute vs. 5 target log-spaced baseline points")

plt.show()

Baseline results plotting

In [ ]:
from train.utils import plot_compute_vs_metric, Plotting

plot_compute_vs_metric("runs_config.csv", Plotting.VALIDATION_ERROR)
plot_compute_vs_metric("runs_config.csv", Plotting.TEST_ERROR)
plot_compute_vs_metric("runs_config.csv", Plotting.IMAGENET)
plot_compute_vs_metric("runs_config.csv", Plotting.MSCOCO_IMG)


Load CLIP model with custom vision and text tower

In [ ]:
from train.utils import precision_to_dtype
from train.open_clip_train.profiler import get_model_flops
from open_clip import create_model

model = create_model("local-dir:model_configs/ViT-B-32", precision="bf16")

get_model_flops(model, 100, dtype=precision_to_dtype("bf16"))


Batch size heuristic based on pareto frontier

In [ ]:

from train.utils import monotonic_bs, query_bs
import numpy as np

samples_seen = np.array([
    1.28e+06, 1.28e+06, 1.28e+06, 3.07e+06, 6.4e+06, 1.28e+07,
    1.28e+07, 3.07e+07, 6.4e+07, 6.4e+07, 1.28e+08, 1.28e+08,
    3.07e+08, 6.4e+08, 3.07e+08, 6.4e+08, 1.28e+09, 6.4e+08,
    6.4e+08, 1.28e+09, 6.4e+08, 3.07e+09, 1.28e+09, 3.07e+09,
    3.07e+09
], dtype=float)

gflops_per_sample = np.array([
    5.51, 11.75, 11.75, 5.51, 5.51, 5.51, 9.74, 5.51, 5.51, 9.74,
    5.51, 11.75, 5.51, 5.51, 14.3, 9.74, 14.54, 43.59, 46.78, 43.59,
    132.37, 43.59, 168.61, 132.37, 168.61
], dtype=float)

batch_size = np.array([
    512, 512, 512, 1024, 1024, 2048, 2048, 4096, 4096, 4096,
    8192, 8192, 16384, 16384, 16384, 32800, 16384, 45056, 32768, 90624,
    45056, 91136, 90624, 91136, 91136
], dtype=int)


total_compute = samples_seen * gflops_per_sample
total_compute = np.round(total_compute, 3)

ranges = monotonic_bs(total_compute, batch_size)
print(query_bs(705280005))


Infer hardware requirements

In [ ]:
import math

global_batch_size = 90624
local_bs = 128
max_gpus_per_node = 4

gpus_needed = global_batch_size // local_bs
gpus_per_node = min(gpus_needed, max_gpus_per_node)
nodes_nr = math.ceil(gpus_needed / max_gpus_per_node)

print(f"GPUs needed:{gpus_needed}")
print(f"GPUs per node:{gpus_per_node}")
print(f"Nodes number:{nodes_nr}")

In [8]:
import neps
root_dir = "results/model=ViT-S-32/seed=0"
full_df, result = neps.status(root_dir)
print(full_df)


    config.grad_clip_norm  config.lr  config.warmup_fraction  config.wd  \
id                                                                        
1                4.615516   0.000003                0.460301   0.000008   
2                1.654799   0.092138                 0.26588   0.032788   
3                9.662704   0.000069                0.181232   0.013194   
4                2.433681   0.000004                0.415114   0.000004   
5                0.662344   0.000003                0.431379   0.000085   
6                4.230629   0.000096                0.224106   0.038049   
7                2.561648   0.152778                0.396517   0.000006   
8                 0.71498   0.000007                0.039153   0.007001   
9                 2.11772   0.000049                0.173994   0.048399   
10                0.69905   0.000047                0.411546   0.005545   

                                             location             state  \
id                      

Max. amount of GFLOPs for our case

In [1]:
from train.utils import precision_to_dtype
from train.open_clip_train.profiler import get_model_flops
from open_clip import create_model

nr_defined_samples = 50000000
model = create_model("local-dir:model_configs/ViT-B-32", precision="bf16")

flops_amount = get_model_flops(model, nr_defined_samples, dtype=precision_to_dtype("bf16"))
print(flops_amount)

nr_defined_samples = 66926609
flops_amount = get_model_flops(model, nr_defined_samples, dtype=precision_to_dtype("bf16"))
print(flops_amount)


C:\Users\Donat\miniconda3\envs\vlm_scaling_law\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module                     FLOP    % Total
--------------------  ---------  ---------
CLIP                  9743.106M    100.00%
 - aten.convolution    154.141M      1.58%
 - aten.addmm         9588.179M     98.41%
 - aten.mm               0.786M      0.01%
 CLIP.transformer     5813.305M     59.67%
  - aten.addmm        5813.305M     59.67%
 CLIP.visual          3929.407M     40.33%
  - aten.convolution   154.141M      1.58%
  - aten.addmm        3774.874M     38.74%
  - aten.mm              0.393M      0.00%
487000000.0
Module                     FLOP    % Total
--------------------  ---------  ---------
CLIP                  9743.106M    100.00%
 - aten.convolution    154.141M      1.58%
 - aten.addmm         9588.179M     98.41%
 - aten.mm               0.786M      0.01%
 CLIP.transformer     5813.305M     59.67%
  - aten.addmm        5813.305M     59.67%
 CLIP.visual          3929.407M     40.33%
  - aten.convolution   154.141M      1.58%
  - aten.addmm        3774.874M     38.74%